# WWII Radio Broadcasts → Transcripts (Whisper) + Error Validation

Downloads public WWII radio broadcasts (FDR Fireside Chats + Churchill wartime addresses) from the **Internet Archive**, transcribes them with **Whisper**, filters low-confidence output, and measures **Word Error Rate** against official transcripts — which it downloads automatically for the FDR chats.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU`.

**Output:** `whisper_output.zip` (per-speech `.txt` + `.json` transcripts and `manifest.csv`). Download it and send it back to Claude for the emotion/frequency analysis.

> Rights: FDR's speeches are public domain (transcripts auto-fetched from the Miller Center). Churchill's texts are under copyright — his audio is transcribed for your own analysis, and you can paste his official transcripts in manually if you want his WER too.

In [ ]:
# 1) Install dependencies  (Colab already has torch, pandas, requests, ffmpeg)
!pip install -q openai-whisper internetarchive jiwer beautifulsoup4

In [ ]:
# 2) Imports and GPU check
import os, re, glob, json, zipfile
from datetime import date
import pandas as pd
import requests
import torch, whisper
from bs4 import BeautifulSoup
from internetarchive import get_item

print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU, then re-run.")

In [ ]:
# 3) Configuration
WHISPER_MODEL = "small.en"   # tiny.en (fastest) < base.en < small.en < medium.en (most accurate, slowest)

WAR_START = date(1939, 9, 1)   # used to auto-filter Churchill's collection to wartime broadcasts
WAR_END   = date(1945, 9, 2)

# FDR Fireside Chats: original broadcasts on the Internet Archive (public domain).
# 'miller_url' is the official transcript used as ground truth for WER; "" = transcribe & analyze but skip WER.
FDR_ITEMS = [
    {"identifier": "2May261940FiresideChat15OnNationalDefenseFDR",        "date": "1940-05-26", "label": "FC15 National Defense",      "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/may-26-1940-fireside-chat-15-national-defense"},
    {"identifier": "3December291940FiresideChat16OnTheArsenalOfDemocracyFDR","date": "1940-12-29", "label": "FC16 Arsenal of Democracy", "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/december-29-1940-fireside-chat-16-arsenal-democracy"},
    {"identifier": "5September111941FiresideChat18OnTheGreerIncidentFDR",   "date": "1941-09-11", "label": "FC18 The Greer Incident",    "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/september-11-1941-fireside-chat-18-greer-incident"},
    {"identifier": "6December91941FiresideChat19OnTheWarWithJapanFRD",      "date": "1941-12-09", "label": "FC19 On the War with Japan", "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/december-9-1941-fireside-chat-19-war-japan"},
    {"identifier": "7February231942FiresideChat20OnTheProgressOfTheWarFDR", "date": "1942-02-23", "label": "FC20 Progress of the War",   "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/february-23-1942-fireside-chat-20-progress-war"},
    {"identifier": "8April281942FiresideChat21OnSacrificeFDR",             "date": "1942-04-28", "label": "FC21 On Sacrifice",         "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/april-28-1942-fireside-chat-21-sacrifice"},
    {"identifier": "9September71942FiresideChat22OnInflationAndFoodPricesFDR","date":"1942-09-07","label": "FC22 Inflation & Food",      "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/september-7-1942-fireside-chat-22-inflation-and-food-prices"},
    {"identifier": "1October121942FiresideChat23OnTheHomeFrontFDR",        "date": "1942-10-12", "label": "FC23 The Home Front",       "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/october-12-1942-fireside-chat-23-home-front"},
    {"identifier": "11May21943FiresideChat24OnTheCoalCrisisFDR",           "date": "1943-05-02", "label": "FC24 The Coal Crisis",      "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/may-2-1943-fireside-chat-24-coal-crisis"},
    {"identifier": "12July281943FiresideChat25OnTheFallOfMussoliniFDR",    "date": "1943-07-28", "label": "FC25 Fall of Mussolini",    "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/july-28-1943-fireside-chat-25-fall-mussolini"},
    {"identifier": "13September81943FiresideChat26OnTheArmisticeInItalyFDR","date": "1943-09-08", "label": "FC26 Armistice in Italy",   "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/september-8-1943-fireside-chat-26-armistice-italy"},
    {"identifier": "15January111944FiresideChat28OnTheStateOfTheUnionFDR",  "date": "1944-01-11", "label": "FC28 State of the Union",   "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/january-11-1944-fireside-chat-28-state-union"},
    {"identifier": "16June51944FiresideChat29OnTheFallOfRomeFDR",          "date": "1944-06-05", "label": "FC29 Fall of Rome",         "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/june-5-1944-fireside-chat-29-fall-rome"},
    {"identifier": "17June121944FiresideChat30OpeningFifthWarLoanDriveFDR", "date": "1944-06-12", "label": "FC30 Fifth War Loan Drive", "miller_url": "https://millercenter.org/the-presidency/presidential-speeches/june-12-1944-fireside-chat-30-opening-fifth-war-loan-drive"},
]

# Churchill wartime broadcasts: ONE Internet Archive item holds many dated .ogg files
# (filenames encode the date as WU<YYMMDD>_...). We download all, then keep only the war years.
CHURCHILL_ITEM = "Winston_Churchill"

AUDIO_DIR, TRANSCRIPT_DIR = "audio", "transcripts"
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

In [ ]:
# 4) Download original audio from the Internet Archive
AUDIO_EXT = (".mp3", ".ogg", ".wav", ".flac", ".m4a")

def download_originals(identifier):
    """Download an item's ORIGINAL audio files (skips IA-generated derivative duplicates)."""
    item = get_item(identifier)
    files = list(item.get_files())
    originals = [f for f in files
                 if getattr(f, "source", "") == "original" and f.name.lower().endswith(AUDIO_EXT)]
    if not originals:  # fallback if the item doesn't tag originals
        originals = [f for f in files if f.name.lower().endswith(AUDIO_EXT)]
    for f in originals:
        dest = os.path.join(AUDIO_DIR, identifier, f.name)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if not os.path.exists(dest):
            print("  downloading:", f.name)
            f.download(file_path=dest)

print("FDR:")
for it in FDR_ITEMS:
    print(" item:", it["identifier"]); download_originals(it["identifier"])
print("Churchill:")
download_originals(CHURCHILL_ITEM)
print("Done downloading.")

In [ ]:
# 5) Build the transcription worklist, tagging each file with leader + date (+ transcript URL for FDR)
def churchill_date_from_name(name):
    """Get the broadcast date from a Churchill filename. Handles ISO names like
    '1940-06-17_BBC_Winston_Churchill_Their_Finest_Hour.mp3' and 'WU<YYMMDD>_...' names."""
    base = os.path.basename(name)
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})", base)          # ISO date anywhere in the filename
    if m:
        y, mm, dd = (int(x) for x in m.groups())
        try: return date(y, mm or 1, dd or 1)
        except ValueError: return None
    m = re.search(r"WU(\d{2})(\d{2})(\d{2})", base)          # fallback: WU<YYMMDD>
    if m:
        yy, mm, dd = (int(x) for x in m.groups())
        try: return date(1900 + yy, mm or 1, dd or 1)
        except ValueError: return None
    return None

def churchill_label_from_name(name):
    """Derive a readable speech title from the filename (helps you match it to a transcript)."""
    raw = os.path.splitext(os.path.basename(name))[0]
    lab = re.sub(r"\d{4}-\d{2}-\d{2}|WU\d+|BBC|Winston|Churchill|_", " ", raw)
    return re.sub(r"\s+", " ", lab).strip() or raw

worklist = []
for it in FDR_ITEMS:                                   # FDR: date + transcript URL from config
    for p in glob.glob(os.path.join(AUDIO_DIR, it["identifier"], "*")):
        if p.lower().endswith(AUDIO_EXT):
            worklist.append({"path": p, "leader": "FDR", "date": it["date"],
                             "label": it["label"], "miller_url": it["miller_url"]})

for p in glob.glob(os.path.join(AUDIO_DIR, CHURCHILL_ITEM, "*")):   # Churchill: date from filename
    if not p.lower().endswith(AUDIO_EXT): continue
    d = churchill_date_from_name(p)
    if d and WAR_START <= d <= WAR_END:
        worklist.append({"path": p, "leader": "Churchill", "date": d.isoformat(),
                         "label": churchill_label_from_name(p), "miller_url": ""})

worklist.sort(key=lambda r: (r["date"], r["leader"]))
print(f"{len(worklist)} recordings queued:")
for r in worklist:
    print(f"  {r['date']}  {r['leader']:9}  {os.path.basename(r['path'])}")

In [ ]:
# 6) Transcribe with Whisper  (the slow step — a few minutes per recording on a GPU)
model = whisper.load_model(WHISPER_MODEL)
print("Loaded Whisper model:", WHISPER_MODEL)

records = []
for i, r in enumerate(worklist, 1):
    stem = f"{r['date']}_{r['leader']}_{i:02d}"
    print(f"[{i}/{len(worklist)}] {stem} ...")
    result = model.transcribe(r["path"], verbose=False)
    text = result["text"].strip()
    segs = result.get("segments", [])

    open(os.path.join(TRANSCRIPT_DIR, stem + ".txt"), "w").write(text)
    json.dump({"stem": stem, "leader": r["leader"], "date": r["date"],
               "label": r["label"], "segments": segs},
              open(os.path.join(TRANSCRIPT_DIR, stem + ".json"), "w"))

    lps = [s.get("avg_logprob", 0.0) for s in segs] or [0.0]
    records.append({"stem": stem, "leader": r["leader"], "date": r["date"], "label": r["label"],
                    "miller_url": r["miller_url"], "model": WHISPER_MODEL, "n_segments": len(segs),
                    "mean_logprob": round(sum(lps) / len(lps), 3), "words": len(text.split())})
print("Transcription complete.")

In [ ]:
# 7) Filter low-confidence segments  (mitigation for noisy 1940s audio)
LOGPROB_THRESHOLD = -1.0   # drop segments whose average log-probability is below this

for rec in records:
    data = json.load(open(os.path.join(TRANSCRIPT_DIR, rec["stem"] + ".json")))
    kept = [s for s in data["segments"] if s.get("avg_logprob", 0.0) >= LOGPROB_THRESHOLD]
    rec["low_conf_dropped"] = len(data["segments"]) - len(kept)
    clean = " ".join(s["text"].strip() for s in kept).strip()
    open(os.path.join(TRANSCRIPT_DIR, rec["stem"] + "_clean.txt"), "w").write(clean)
print("Wrote *_clean.txt (low-confidence segments removed).")

In [ ]:
# 8) Measure accuracy: Word Error Rate vs official transcripts
#    FDR transcripts are fetched automatically from the Miller Center (public domain).
import jiwer

def normalize(t):
    t = re.sub(r"\([^)]*\)", " ", t)              # drop editorial brackets like "(remind)"
    t = re.sub(r"[^a-z0-9\s]", " ", t.lower())     # keep letters/digits/space
    return re.sub(r"\s+", " ", t).strip()

def wer(reference, hypothesis):
    return jiwer.wer(normalize(reference), normalize(hypothesis))

def fetch_miller_transcript(url):
    """Download an FDR speech transcript from the Miller Center page."""
    try:
        html = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"}).text
    except Exception as e:
        print("   fetch failed:", e); return ""
    soup = BeautifulSoup(html, "html.parser")
    node = soup.find(id="dp-expandable-text") or soup.select_one(".transcript, article")
    return node.get_text(" ", strip=True) if node else ""

# Churchill references: drop transcripts into a 'references/' folder. Filenames can be in ANY common
# date format (1940-06-04.txt, 18_June_1940.txt, February_9__1941.txt, ...) - the date is parsed out.
# Each is matched to the recording with the NEAREST date (within a few days), since the archive's audio
# is sometimes filed a day off the delivery date. FDR is auto-fetched above; Churchill texts are
# copyrighted, so you supply a few yourself for fair-use validation.
REF_DIR = "references"
try:
    from dateutil import parser as _dtp
    def date_from_filename(s):
        s = re.sub(r"\.txt$", "", s, flags=re.I)
        try: return _dtp.parse(re.sub(r"_+", " ", s), fuzzy=True).date()
        except Exception:
            m = re.search(r"(\d{4})-(\d{2})-(\d{2})", s)
            return date(*map(int, m.groups())) if m else None
except Exception:
    def date_from_filename(s):
        m = re.search(r"(\d{4})-(\d{2})-(\d{2})", s)
        return date(*map(int, m.groups())) if m else None

ref_list = []  # (date, text, filename)
if os.path.isdir(REF_DIR):
    for fp in glob.glob(os.path.join(REF_DIR, "*.txt")):
        d = date_from_filename(os.path.basename(fp))
        if d: ref_list.append((d, open(fp, encoding="utf-8", errors="ignore").read(), os.path.basename(fp)))
print(f"Loaded {len(ref_list)} Churchill reference file(s) from {REF_DIR}/")

def nearest_churchill_ref(rec_date_iso, tol_days=4):
    rd = date.fromisoformat(rec_date_iso)
    cand = sorted((abs((d - rd).days), txt, fn) for d, txt, fn in ref_list)
    return (cand[0][1], cand[0][2]) if cand and cand[0][0] <= tol_days else ("", "")

print("sanity WER (expect 0.2):", round(wer("we shall fight on the beaches", "we will fight on the beaches"), 3))
for rec in records:
    src = ""
    if rec["leader"] == "FDR" and rec.get("miller_url"):
        ref = fetch_miller_transcript(rec["miller_url"])
    elif rec["leader"] == "Churchill":
        ref, src = nearest_churchill_ref(rec["date"])
    else:
        ref = ""
    if ref:
        hyp = open(os.path.join(TRANSCRIPT_DIR, rec["stem"] + ".txt")).read()
        rec["wer"] = round(wer(ref, hyp), 3)
        print(f"    {rec['stem']}: WER = {rec['wer']}" + (f"  <- {src}" if src else f"  (ref words {len(normalize(ref).split())})"))
    else:
        rec["wer"] = None

# Flag any Churchill reference files that didn't line up with a recording
for d, _t, fn in ref_list:
    if not any(r["leader"] == "Churchill" and abs((date.fromisoformat(r["date"]) - d).days) <= 4 for r in records):
        print(f"  note: '{fn}' matched no recording (no audio near that date) - skipping it")

scored = [r["wer"] for r in records if r["wer"] is not None]
if scored:
    print(f"\nValidated {len(scored)} speeches; overall WER = {round(sum(scored)/len(scored), 3)}")
else:
    print("\nNo references matched yet (add Churchill files to references/, or check the FDR fetch).")

In [ ]:
# 9) Package everything and download
manifest = pd.DataFrame(records).sort_values(["date", "leader"])
manifest.to_csv("manifest.csv", index=False)
print(manifest.to_string(index=False))

with zipfile.ZipFile("whisper_output.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("manifest.csv")
    for f in glob.glob(os.path.join(TRANSCRIPT_DIR, "*")):
        z.write(f)

from google.colab import files
files.download("whisper_output.zip")
print("Downloaded whisper_output.zip")

## Next steps

1. **Send `whisper_output.zip` back to Claude** — the `manifest.csv` (with WER + confidence per speech) and the `*_clean.txt` transcripts feed the emotion/frequency analysis and the interactive timeline.
2. **FDR error rates are computed automatically** (transcripts fetched from the Miller Center). To add **Churchill** WER, drop each transcript as a plain-text file into a `references/` folder, with the date in the filename in any common format (`1940-06-04.txt`, `18_June_1940.txt`, `February_9__1941.txt` all work), then re-run cell 8. No pasting into code.
3. **To add even more recordings**, drop Internet Archive identifiers into `FDR_ITEMS` (cell 3). The discovery helper below lists candidates.

In [ ]:
# (Optional) Discover more Internet Archive audio identifiers to add to FDR_ITEMS
from internetarchive import search_items
for hit in search_items('subject:"fireside chat" AND mediatype:audio'):
    print(hit["identifier"])